In [46]:
import pandas as pd
import numpy as np
import joblib

In [47]:
# Load dataset
df = pd.read_excel(r"C:\Users\yapen\Desktop\FinalTestDataset2025.xls")

# Load saved models and outlier parameters
gene_clf = joblib.load("gene_classifier.pkl")
outlier_info = joblib.load("outlier_params.pkl")
rf_model = joblib.load("rfs_rf_model.pkl")

# Extract quartile information for outlier detection
Q1 = outlier_info["Q1"]
Q3 = outlier_info["Q3"]
IQR = Q3 - Q1

# Keep test IDs separately, remove ID column from feature set
test_id = df["ID"].copy()
df = df.drop(columns=["ID"])


In [48]:
df = df.replace(999, np.nan)
print(f'Total Number of Missing Value is {df.isna().sum().sum()}')

print(f'Total Number of Row with Missing value is {(df.isna().any(axis=1)).sum()}')

print(f'Total Number of Column with Missing value is {(df.isna().any(axis=0)).sum()}')

Total Number of Missing Value is 29
Total Number of Row with Missing value is 24
Total Number of Column with Missing value is 7


In [49]:
# Recode HistologyType: convert {1 → 0, 2 → 1}
df["HistologyType"] = df["HistologyType"].map({1: 0, 2: 1})

# Specify ordinal variables
ordinal_cols = ["TumourStage", "Proliferation", "ChemoGrade"]

# Ensure ordinal columns use integer dtype (nullable Int64 for safety)
df[ordinal_cols] = df[ordinal_cols].astype("Int64")


In [50]:
print(f'Total Number of Missing Value is {df.isna().sum().sum()}')
print(f"The number of rows without LNStatus = {df['LNStatus'].isna().sum()}")
print(f"The number of rows without Gene = {df['Gene'].isna().sum()}")

Total Number of Missing Value is 29
The number of rows without LNStatus = 3
The number of rows without Gene = 20


In [51]:
# Column containing the gene label
gene_col = "Gene"

# Numerical feature columns (exclude the gene label itself)
feat_cols = df.select_dtypes(include="number").columns.drop(gene_col)

# Identify rows where the gene value is missing
mask_test_missing = df[gene_col].isna()

# If any gene values are missing, predict them using the trained classifier
if mask_test_missing.sum() > 0:
    df.loc[mask_test_missing, gene_col] = gene_clf.predict(
        df.loc[mask_test_missing, feat_cols]
    )

    
print(f'Total Number of Missing Value is {df.isna().sum().sum()}')
print(f"The number of rows without LNStatus = {df['LNStatus'].isna().sum()}")
print(f"The number of rows without Gene = {df['Gene'].isna().sum()}")

Total Number of Missing Value is 9
The number of rows without LNStatus = 3
The number of rows without Gene = 0


In [52]:
# Fill missing values for all non-object (numeric) columns using the median
for col in df.columns:
    if df[col].dtype != "object":   # Only numeric columns
        median_value = df[col].median()
        df[col] = df[col].fillna(median_value)

print(f'Total Number of Missing Value is {df.isna().sum().sum()}')
print(f"The number of rows without LNStatus = {df['LNStatus'].isna().sum()}")
print(f"The number of rows without Gene = {df['Gene'].isna().sum()}")

Total Number of Missing Value is 0
The number of rows without LNStatus = 0
The number of rows without Gene = 0


In [53]:
test_clip = df.clip(
    lower=Q1 - 1.5 * IQR,
    upper=Q3 + 1.5 * IQR,
    axis=1
)

In [54]:
# Generate predictions using the trained random forest model
y_pred = np.round(rf_model.predict(test_clip), 4)

# Create output DataFrame with IDs and predicted survival outcomes
out = pd.DataFrame({
    "ID": test_id.values,
    "RelapseFreeSurvival (outcome)": y_pred
})

# Save predictions to a CSV file
out.to_csv("RFSPrediction.csv", index=False)

print("Saved RFSPrediction.csv")


Saved RFSPrediction.csv
